# Week 5 — Simple Model Comparison

Train Decision Tree and Random Forest regressors, compare their test R² with
the saved Linear Regression baseline, and document model behavior.

Run Notebook 02 and Notebook 03 first. Use the same four prepared CSVs that
were used to create `baseline_results.csv`. If you change the prepared data,
rerun Notebook 03 before comparing. Keep this notebook in `notebooks/`.
Preprocessing and the chronological split are already handled by Notebook 02.

## 1. Import libraries

Load pandas, the two additional regressors, and the R² metric.

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

## 2. Load the prepared data and saved baseline

Use the existing training/test files without changing the rows or their order.
Load the Linear Regression score recorded by Notebook 03.

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv")["ClosePrice"]
y_test = pd.read_csv("../data/processed/y_test.csv")["ClosePrice"]

baseline_results = pd.read_csv("../data/processed/baseline_results.csv")
baseline = baseline_results.loc[
    baseline_results["Model"].eq("Linear Regression")
].iloc[0]
baseline_r2 = baseline["Test_R2"]

print(f"Linear Regression baseline R²: {baseline_r2:.4f}")

Linear Regression baseline R²: 0.8158


## 3. Sanity checks

Check aligned input/target lengths, identical feature columns, and no missing
predictors. Compare dataset sizes with the saved baseline metadata to catch an
obviously stale baseline. These size checks cannot prove that the same records
were used; keep the original CSVs unchanged for the comparison.

In [3]:
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert X_train.columns.equals(X_test.columns)
assert not X_train.isna().any().any()
assert not X_test.isna().any().any()
assert len(X_train) == baseline["TrainingRows"]
assert len(X_test) == baseline["TestRows"]
assert X_train.shape[1] == baseline["PredictorColumns"]

print("Data checks passed.")

Data checks passed.


## 4. Train and evaluate Decision Tree

A Decision Tree learns splits such as area or location thresholds. Limit depth
to 18 and require at least five training records per leaf to restrain complexity.
These are initial fixed settings, not tuned values. `random_state=42` makes the
experiment repeatable. Compare training and test R² to help recognize overfitting.

In [4]:
tree_model = DecisionTreeRegressor(
    max_depth=18,
    min_samples_leaf=5,
    random_state=42
)

tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)

tree_train_r2 = tree_model.score(X_train, y_train)
tree_test_r2 = r2_score(y_test, tree_predictions)

print(f"Decision Tree training R²: {tree_train_r2:.4f}")
print(f"Decision Tree test R²: {tree_test_r2:.4f}")

Decision Tree training R²: 0.9203
Decision Tree test R²: 0.8023


## 5. Train and evaluate Random Forest

Random Forest averages the predictions of 100 trees. This can reduce the
instability of a single tree, but requires more computation. Use the same depth
and leaf limits, a fixed seed, and available CPU cores for parallel training.
The test data is passed only to predict/evaluate, not to fit.

In [5]:
forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)

forest_train_r2 = forest_model.score(X_train, y_train)
forest_test_r2 = r2_score(y_test, forest_predictions)

print(f"Random Forest training R²: {forest_train_r2:.4f}")
print(f"Random Forest test R²: {forest_test_r2:.4f}")

Random Forest training R²: 0.9264
Random Forest test R²: 0.8664


## 6. Compare against the baseline and save results

Higher test R² is better. A positive change means a higher R² than the saved
Linear Regression baseline on this test set. The change is an absolute R²
difference, not a percentage of correctly priced properties.

In [6]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "Test_R2": [baseline_r2, tree_test_r2, forest_test_r2]
})

comparison["ChangeFromBaseline"] = comparison["Test_R2"] - baseline_r2

display(comparison.round(4))
comparison.to_csv("../data/processed/model_comparison.csv", index=False)

,Model,Test_R2,ChangeFromBaseline
0,Linear Regression,0.8158,0.0000
1,Decision Tree,0.8023,-0.0135
2,Random Forest,0.8664,0.0506


## 7. Model behavior and findings

- **Decision Tree:** captures nonlinear relationships and interactions. Its
  predictions come from individual leaves. A substantially higher training R²
  than test R² suggests possible overfitting or a difference between the periods.
- **Random Forest:** averages many trees and often generalizes better than one
  tree. It takes longer to train and is less straightforward to interpret.
- Compare both test R² scores with the saved Linear Regression baseline.
  A model has not improved the baseline merely because its training score is high.
- Record the scores printed above and describe which model had the higher test
  R² and which had the larger training/test gap. A gap is diagnostic evidence,
  not proof of its cause.

This is a fixed model comparison using the existing split. Use earlier
validation data if you later tune these starting parameter values.

### Observed in the verification run

- Decision Tree: training R² **0.9203**, test R² **0.8023**; gap **0.1180**.
- Random Forest: training R² **0.9264**, test R² **0.8664**; gap **0.0600**.
- Linear Regression reference: test R² **0.8158**.

- Random Forest had the highest test R², improving on the baseline by **0.0506**.
  The Decision Tree scored **0.0135** below the baseline.
- Both models fit the training data better than the test data. The Decision
  Tree had the larger gap, suggesting greater overfitting or sensitivity to
  differences between the months. Averaging trees helped the forest generalize
  better in this run.

These saved outputs use reconstructed CSVs from the uploaded raw data and current preprocessing notebook. Your local prepared files were not uploaded; rerun with your files to record your own results.
